# [AUDIT DATA PRA-PELATIHAN] Pemeriksaan Shortcut Sumber & Area di Luar Paru

Notebook pendukung ini dijalankan **setelah Notebook 02** (setelah cache `I_224` dan `M_224` selesai) dan **sebelum Notebook 04** (pelatihan utama EfficientNet-B0):
1. **Probe 1 (Tebak Sumber):** Mengukur seberapa mudah citra Kermany vs RSNA dibedakan. Nilai tinggi membuktikan perlunya penyeimbangan kelas 1:1 di masing-masing sumber.
2. **Probe 2 (Area di Luar Paru, $I \odot (1 - M)$):** Mengukur AUC prediksi kelas Normal vs Pneumonia **hanya dari area di luar paru** untuk memeriksa keberadaan informasi latar belakang (*shortcut*).

Hasil audit disimpan di `outputs/tables/audit/shortcut_audit_summary.csv` dan tidak memodifikasi data maupun konfigurasi eksperimen utama.

In [ ]:
# Sel 1: Setup lingkungan & muat metadata cache lung mask
import json
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

for candidate in [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content/drive/MyDrive/LAB_TA_GradCAM_Pneumonia/code"),
    Path("/content/gradcam-pneumonia"),
]:
    if (candidate / "src" / "paths.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

import pandas as pd
from IPython.display import display

from exploratory.shortcut_audit import run_shortcut_audit_probes
from src.paths import ensure_output_dirs, load_config
from src.seed import set_global_seed

CFG = load_config()
PATHS = ensure_output_dirs(cfg=CFG)
SEED = int(CFG["assumed"]["split_seed"]["value"])
set_global_seed(SEED, deterministic=True)

meta_csv = PATHS["processed"] / "lung_mask_metadata.csv"
assert meta_csv.exists(), f"Cache metadata belum tersedia di {meta_csv}. Jalankan Notebook 02 terlebih dahulu!"
meta_df = pd.read_csv(meta_csv)
print(f"Metadata dimuat: {len(meta_df)} citra | Sumber: {meta_df['source'].value_counts().to_dict()}")

In [ ]:
# Sel 2: Jalankan Probe 1 (Tebak Sumber) & Probe 2 (Area di Luar Paru) + GATE CHECK
audit_res = run_shortcut_audit_probes(
    meta_df=meta_df,
    processed_dir=PATHS["processed"],
    seed=SEED,
    max_train_samples=3000,
)

summary_df = audit_res["summary_df"]
out_csv = PATHS["outputs"] / "tables" / "audit" / "shortcut_audit_summary.csv"
out_json = PATHS["outputs"] / "tables" / "audit" / "shortcut_audit_report.json"
summary_df.to_csv(out_csv, index=False, encoding="utf-8")
out_json.write_text(
    json.dumps({k: v for k, v in audit_res.items() if k != "summary_df"}, indent=2),
    encoding="utf-8",
)

print("===================================================================")
print("              GATE CHECK — AUDIT SHORTCUT PRA-PELATIHAN            ")
print("===================================================================")
display(summary_df)
print(f"\nLaporan tersimpan di: {out_csv}")
print("Interpretasi singkat:")
print("- Probe 1 (Tebak Sumber) tinggi -> membuktikan perbedaan karakter Kermany vs RSNA nyata, sehingga rasio 1:1 per sumber wajib dijaga.")
print("- Probe 2 (Area Luar Paru) AUC > 0.60 -> menunjukkan keberadaan sinyal luar paru (confounder), memperkuat relevansi pengujian fusi 4-channel.")